In [15]:
import numpy as np
import pandas as pd
px = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\1. Raw Data\prices.csv")

In [16]:
px["date"] = pd.to_datetime(px["date"])
px = px.sort_values("date")

px["Coal_M1"] = px["Coal_M1"].ffill(limit=3)  # 44 holiday blanks;
px = px.dropna(subset=["TTF_M1", "TTF_M2", "TTF_M3"]).copy()

px = px.rename(columns={"TTF_M1": "m1", "TTF_M2": "m2", "TTF_M3": "m3"})
px["s12"] = px["m1"] - px["m2"]
px["s13"] = px["m1"] - px["m3"]
px["s23"] = px["m2"] - px["m3"]
px["s12_pct"] = px["s12"] / px["m2"] * 100
px["s13_pct"] = px["s13"] / px["m3"] * 100
px["s23_pct"] = px["s23"] / px["m3"] * 100
px["s23_pct"] = px["s23"] / px["m3"] * 100

In [17]:
month_starts = pd.date_range(px["date"].min().replace(day=1), px["date"].max() + pd.offsets.MonthBegin(2), freq="MS")
expiry = [pd.bdate_range(end=ms - pd.Timedelta(days=1), periods=2)[0] for ms in month_starts]
exp = pd.DataFrame({"expiry": expiry, "delivery": month_starts})
px = pd.merge_asof(px, exp.sort_values("expiry"), left_on="date", right_on="expiry", direction="forward")

changed = px["delivery"].ne(px["delivery"].shift())
changed.iloc[0] = False
px["roll_day"] = changed.astype(int)  # 1 on the first session of the new front month
px["dte"] = px.groupby("delivery").cumcount(ascending=False)

px["m1_contract"] = pd.NA
px["m2_contract"] = pd.NA
px["m3_contract"] = pd.NA

for month, name in enumerate(["d_feb", "d_mar", "d_apr", "d_may", "d_jun", "d_jul", "d_aug", "d_sep", "d_oct", "d_nov", "d_dec"], start=2):
    px[name] = (px["delivery"].dt.month == month).astype(int)  # January is the base month

px["inj_season"] = px["date"].dt.month.between(4, 9).astype(int)
px["crisis"] = px["date"].between("2021-07-01", "2023-03-31").astype(int)

In [18]:
for price, vol in [("m1", "vol_m1"), ("m2", "vol_m2"), ("m3", "vol_m3")]:
    ret = np.log(px[price] / px[price].shift(1))
    ret = ret.mask(px["roll_day"].eq(1))  # this return changes contract
    px[vol] = ret.rolling(20, min_periods=20).std() * np.sqrt(252) * 100

for spread, sd in [("s12", "sd_ds12"), ("s13", "sd_ds13")]:
    change = px[spread].diff().mask(px["roll_day"].eq(1))
    px[sd] = change.rolling(20, min_periods=20).std()

roll_ahead = px["roll_day"].rolling(5).sum().shift(-5)  # a roll on any of the next 5 sessions
px["ds12_fwd5"] = (px["s12"].shift(-5) - px["s12"]).mask(roll_ahead.gt(0))
px["ds13_fwd5"] = (px["s13"].shift(-5) - px["s13"]).mask(roll_ahead.gt(0))

In [19]:
coal_eur = px["Coal_M1"] / px["EUR/USD"] / 6.978  # EUR/MWh thermal
px["ttf_switch"] = (0.52 / 0.38) * (coal_eur + 0.341 * px["EUA_M1"]) - 0.202 * px["EUA_M1"]
px["switch_gap"] = px["m1"] - px["ttf_switch"]

In [20]:
px = px[[
    "date", "m1_contract", "m2_contract", "m3_contract", "roll_day", "dte",
    "m1", "m2", "m3", "s12", "s13", "s23", "s12_pct", "s13_pct", "s23_pct",
    "ds12_fwd5", "ds13_fwd5",
    "vol_m1", "vol_m2", "vol_m3", "sd_ds12", "sd_ds13",
    "ttf_switch", "switch_gap",
    "d_feb", "d_mar", "d_apr", "d_may", "d_jun", "d_jul", "d_aug", "d_sep", "d_oct", "d_nov", "d_dec",
    "inj_season", "crisis",
]]

In [21]:
px.to_csv("prices_processed.csv", index=False)